# 📈 範例 01：廣告效益預測 (迴歸篇) — AI 協作實戰筆記本

本筆記本完整重現按照 `README.md` 向 AI 提問並逐步執行的成果。初學者可以一邊對照提示詞，一邊觀察每一步的輸入、輸出與圖表解讀。

## 階段 0：環境準備與中文字型設定

In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 設定繁體中文字型
try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "sans-serif"]
    plt.rcParams["axes.unicode_minus"] = False

print("環境套件與字型載入完成！")

## 階段 1：資料載入與探索性資料分析 (EDA)

In [ ]:
# 載入資料集
df = pd.read_csv("data/advSale.csv")
print("=== 資料結構摘要 ===")
df.info()
display(df.head())

In [ ]:
# 計算特徵相關係數
num_cols = ["電視廣告", "廣播廣告", "社群媒體廣告", "銷售額"]
corr = df[num_cols].corr()
print("=== 各廣告與銷售額之相關係數 ===")
print(corr["銷售額"].sort_values(ascending=False))

# 繪製熱力圖
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title("廣告投入管道與銷售額相關性熱力圖", fontsize=14)
plt.tight_layout()
plt.show()

## 階段 2：資料前處理與 Pipeline 建構 (防資料洩漏)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 切分特徵 X 與目標 y
X = df.drop(columns=["銷售額"])
y = df["銷售額"]

# 區分數值特徵與類別特徵
numeric_features = ["電視廣告", "廣播廣告", "社群媒體廣告"]
categorical_features = ["網紅廣告"]

# 定義 ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical_features)
    ]
)

# 切分訓練集 (80%) 與測試集 (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"訓練集樣本數: {X_train.shape[0]}, 測試集樣本數: {X_test.shape[0]}")

## 階段 3：模型選型與交叉驗證比較 (Baseline vs. Advanced)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# 建立 Pipeline
model_lr = Pipeline([
    ("prep", preprocessor),
    ("regressor", LinearRegression())
])

model_rf = Pipeline([
    ("prep", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

# 5 折交叉驗證 (以 R2 評估)
scores_lr = cross_val_score(model_lr, X_train, y_train, cv=5, scoring="r2")
scores_rf = cross_val_score(model_rf, X_train, y_train, cv=5, scoring="r2")

print(f"[多元線性迴歸] 5折交叉驗證平均 R2: {scores_lr.mean():.4f} (標準差: {scores_lr.std():.4f})")
print(f"[隨機森林迴歸] 5折交叉驗證平均 R2: {scores_rf.mean():.4f} (標準差: {scores_rf.std():.4f})")

## 階段 4：測試集評估與指標解讀

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 擬合訓練集
model_lr.fit(X_train, y_train)
model_rf.fit(X_train, y_train)

# 在測試集上評估
y_pred_lr = model_lr.predict(X_test)
y_pred_rf = model_rf.predict(X_test)

results = pd.DataFrame({
    "評估指標": ["MAE (平均絕對誤差)", "RMSE (均方根誤差)", "R² (判定係數)"],
    "多元線性迴歸": [
        mean_absolute_error(y_test, y_pred_lr),
        root_mean_squared_error(y_test, y_pred_lr),
        r2_score(y_test, y_pred_lr)
    ],
    "隨機森林迴歸": [
        mean_absolute_error(y_test, y_pred_rf),
        root_mean_squared_error(y_test, y_pred_rf),
        r2_score(y_test, y_pred_rf)
    ]
})
display(results)

In [ ]:
# 視覺化：真實銷售額 vs 預測銷售額
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_lr, alpha=0.5, color="royalblue", label="線性迴歸預測點")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2, label="完美預測線 (y=x)")
plt.title("測試集預測效果評估 (真實值 vs. 預測值)", fontsize=14)
plt.xlabel("實際銷售額", fontsize=12)
plt.ylabel("模型預測銷售額", fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

## 階段 5：實務情境推論試算 (Inference)

In [ ]:
def predict_sales(pipeline, plan_dict):
    """輸入新廣告配置方案，回傳預測銷售額"""
    input_df = pd.DataFrame([plan_dict])
    pred = pipeline.predict(input_df)[0]
    return pred

# 測試方案 A 與方案 B
plan_A = {"電視廣告": 80.0, "廣播廣告": 30.0, "社群媒體廣告": 5.0, "網紅廣告": "大於百萬"}
plan_B = {"電視廣告": 50.0, "廣播廣告": 10.0, "社群媒體廣告": 2.0, "網紅廣告": "小於一萬"}

pred_A = predict_sales(model_lr, plan_A)
pred_B = predict_sales(model_lr, plan_B)

print(f"【方案 A 預估銷售額】: {pred_A:.2f} 萬元")
print(f"【方案 B 預估銷售額】: {pred_B:.2f} 萬元")
print(f"➡️ 方案 A 比 方案 B 預期多帶來 {pred_A - pred_B:.2f} 萬元的銷售增長！")